# FINPLE KR Price Metrics + Raw Daily Colab

Step 114-2Y extends the existing KR yfinance collector. Provider calls run only in this user-operated Colab. Six-digit KR ticker identity is preserved through 20-asset smoke, resumable 100-asset chunks, and combine.


In [ ]:
COLLECTION_REF = "codex/step114-2y-one-click-full-universe-app-preview"
!rm -rf /content/FINPLE
!git clone --depth 1 https://github.com/vip930sw/FINPLE.git /content/FINPLE
%cd /content/FINPLE
!git fetch --depth 1 origin {COLLECTION_REF}
!git checkout {COLLECTION_REF}
!pip -q install yfinance pandas


In [ ]:
from datetime import date, datetime, timezone
from pathlib import Path
import csv, json, subprocess

AS_OF = date.today().isoformat()
RUN_TAG = AS_OF.replace('-', '')
RETRIEVED_AT = datetime.now(timezone.utc).replace(microsecond=0).isoformat()
UNIVERSE = Path('src/data/tickers/finple_app_candidates_6000_balanced_v1.csv')
ROOT = Path('/content/finple-step114-2y-kr') / RUN_TAG
SMOKE_DIR = ROOT / 'smoke'
CHUNK_DIR = ROOT / 'chunks'
COMBINED_DIR = ROOT / 'combined'
for path in [SMOKE_DIR, CHUNK_DIR, COMBINED_DIR]:
    path.mkdir(parents=True, exist_ok=True)

with UNIVERSE.open('r', encoding='utf-8-sig', newline='') as handle:
    universe_rows = list(csv.DictReader(handle))
KR_TOTAL = sum(row['market'] == 'KR' for row in universe_rows)
print({'asOf': AS_OF, 'retrievedAt': RETRIEVED_AT, 'krCandidates': KR_TOTAL, 'root': str(ROOT)})


## 1. Required 20-asset smoke


In [ ]:
smoke_prefix = SMOKE_DIR / 'kr_0000_0020'
subprocess.run([
    'python', 'scripts/build_kr_price_metrics_overlay_chunked.py',
    '--input', str(UNIVERSE),
    '--out-runtime', str(smoke_prefix) + '_metrics.csv',
    '--out-audit', str(smoke_prefix) + '_audit.csv',
    '--out-summary', str(smoke_prefix) + '_summary.json',
    '--out-raw', str(smoke_prefix) + '_raw_daily.csv',
    '--as-of', AS_OF, '--start', '0', '--limit', '20',
    '--checkpoint-every', '5', '--retrieved-at', RETRIEVED_AT, '--resume',
], check=True)
with open(str(smoke_prefix) + '_summary.json', encoding='utf-8') as handle:
    smoke_summary = json.load(handle)
assert smoke_summary['processed_count'] == 20
assert smoke_summary['raw_daily_asset_count'] == 20, smoke_summary
with open(str(smoke_prefix) + '_raw_daily.csv', encoding='utf-8-sig', newline='') as handle:
    smoke_raw = list(csv.DictReader(handle))
assert all(len(row['ticker']) == 6 and row['ticker'].isdigit() for row in smoke_raw)
print(json.dumps(smoke_summary, indent=2))


## 2. Full KR collection in resumable 100-asset chunks


In [ ]:
for start in range(0, KR_TOTAL, 100):
    end = min(start + 100, KR_TOTAL)
    prefix = CHUNK_DIR / f'kr_{start:04d}_{end:04d}'
    subprocess.run([
        'python', 'scripts/build_kr_price_metrics_overlay_chunked.py',
        '--input', str(UNIVERSE),
        '--out-runtime', str(prefix) + '_metrics.csv',
        '--out-audit', str(prefix) + '_audit.csv',
        '--out-summary', str(prefix) + '_summary.json',
        '--out-raw', str(prefix) + '_raw_daily.csv',
        '--as-of', AS_OF, '--start', str(start), '--limit', str(end - start),
        '--checkpoint-every', '25', '--retrieved-at', RETRIEVED_AT, '--resume',
    ], check=True)
print('KR chunks complete:', len(list(CHUNK_DIR.glob('*_metrics.csv'))))


## 3. Combine and reconcile


In [ ]:
KR_METRICS = COMBINED_DIR / 'kr_price_metrics_overlay.csv'
KR_RAW = COMBINED_DIR / 'kr_raw_daily_prices.csv'
KR_SUMMARY = COMBINED_DIR / 'kr_price_metrics_summary.json'
subprocess.run([
    'python', 'scripts/combine_kr_price_metrics_chunks.py',
    '--pattern', str(CHUNK_DIR / '*_metrics.csv'),
    '--out-runtime', str(KR_METRICS), '--out-summary', str(KR_SUMMARY),
    '--raw-pattern', str(CHUNK_DIR / '*_raw_daily.csv'), '--out-raw', str(KR_RAW),
], check=True)
with KR_SUMMARY.open(encoding='utf-8') as handle:
    combined_summary = json.load(handle)
assert combined_summary['row_count'] == KR_TOTAL
assert combined_summary['rawDailyAssetCount'] <= KR_TOTAL
print(json.dumps(combined_summary, indent=2))
print('One-Click inputs:', KR_RAW, KR_METRICS)


In [ ]:
DOWNLOAD_OUTPUTS = False
if DOWNLOAD_OUTPUTS:
    from google.colab import files
    for target in [KR_RAW, KR_METRICS, KR_SUMMARY]:
        files.download(str(target))
else:
    print('Set DOWNLOAD_OUTPUTS=True after checking reconciliation.')
